# Simulate all policies
This notebook simulated all candidates policies from 2012-2024.

In [ ]:
# imports
import iot
from iot import iot_user
import numpy as np
import pandas as pd
import json
import plotly.express as px
import os
from plotly.colors import n_colors
from scipy.interpolate import UnivariateSpline
import plotly.graph_objects as go

# set plotly template
template = "plotly_white"
COLOR_SEQUENCE = ["red", "blue"]  # Republican, Democrat
dash_sequence = ['dash', 'dot', 'dashdot', "solid"]  # 2012, 2016, 2020

# Create path for plots to be saved to
CUR_DIR = os.getcwd()
path = os.path.join(CUR_DIR, "plots")
if not os.path.exists(path):
    os.makedirs(path)

In [ ]:
# Read in candidate platform JSON files
obama2015_path = "https://raw.githubusercontent.com/jdebacker/examples/pres_proposals/psl_examples/taxcalc/Obama2015.json"
romney2012_path = "https://raw.githubusercontent.com/jdebacker/examples/pres_proposals/psl_examples/taxcalc/Romney2012.json"
clinton2016_path = "https://raw.githubusercontent.com/jdebacker/examples/pres_proposals/psl_examples/taxcalc/Clinton2016.json"
trump2016_path = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/Trump2016.json"
biden2020_path = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/Biden2020.json"
trump2020_path = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/TCJA.json"
harris2024_path = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/Harris2024.json"
trump2024_path = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/Trump2024.json"

pre_2020_baseline = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/2017_law.json"
baseline_2020 = "https://raw.githubusercontent.com/PSLmodels/examples/main/psl_examples/taxcalc/TCJA.json"


candidate_dict = {
    "Obama 2015": {"policy_path": obama2015_path, "baseline_path": [pre_2020_baseline], "start_year": 2016},
    "Romney 2012": {"policy_path": romney2012_path, "baseline_path": [pre_2020_baseline], "start_year": 2014}, #wanted to do 13,  but taxcalc with CPS only goes to 14
    "Clinton 2016": {"policy_path": clinton2016_path, "baseline_path": [pre_2020_baseline], "start_year": 2017},
    "Trump 2016": {"policy_path": trump2016_path, "baseline_path": [pre_2020_baseline], "start_year": 2017},
    "Biden 2020": {"policy_path": biden2020_path, "baseline_path": [pre_2020_baseline, baseline_2020], "start_year": 2021},
    "Trump 2020": {"policy_path": trump2020_path, "baseline_path": [pre_2020_baseline], "start_year": 2021},
    # "Harris 2024": {"policy_path": harris2024_path, "baseline_path": [pre_2020_baseline, baseline_2020], "start_year": 2025},
    # "Trump 2024": {"policy_path": trump2024_path, "baseline_path": [pre_2020_baseline, baseline_2020], "start_year": 2025}
    "Harris 2024": {"policy_path": harris2024_path, "baseline_path": None, "start_year": 2026},  # Use 2026 for these candidates since largest diff is how treat TCJA expirations
    "Trump 2024": {"policy_path": trump2024_path, "baseline_path": None, "start_year": 2026}
    }

In [ ]:
# Create IOT objects for each candidate platform
policies = []
baseline_policies = []
labels = list(candidate_dict.keys())
# get years from start_year in candidate_dict
years = [v["start_year"] for v in candidate_dict.values()]
for k, v in candidate_dict.items():
    # with open(v["policy_path"], "r") as file:
        # json1 = file.read()
    json1 = v["policy_path"]#json.load(open(v["policy_path"]))
    print(json1)
    policies.append(json1)
    baseline_policies.append(v["baseline_path"])

#income_measure = "expanded_income"
income_measure = "e00200" # default

iot_all = iot_user.iot_comparison(
    policies=policies,
    baseline_policies=baseline_policies,
    mtr_smoother="kreg",
    labels=labels,
    years=years,
    data="CPS",
    income_measure=income_measure,
    dist_type="Pln",
    #mtr_wrt=income_measure
)

In [ ]:
iot_all.iot[-1].mtr

In [ ]:
iot_all.iot[-2].mtr

In [ ]:
len(iot_all.iot)

In [ ]:
# Plots of f(z) for each year/candidate
fplot = iot_all.plot(var="f")
fplot.update_layout(
    template=template,
)
fplot.write_image(
            os.path.join(path, "income_dist.png")
        )

In [ ]:
# Plot distribution f(z) with histogram
z_max = 300_000
data = iot_all.iot[0].data_original
data_capped = data[data[income_measure] <= z_max]

# Create weighted histogram
hist, bin_edges = np.histogram(
    data_capped[income_measure], bins=15, weights=data_capped['s006'], density=True
)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# Create figure
fig = go.Figure()
fig.add_trace(go.Bar(
    x=bin_centers, y=hist, name='Data', opacity=0.6,
    width=bin_edges[1] - bin_edges[0]
))

# Add fitted distribution
iot_obj = iot_all.iot[0]
mask = iot_obj.z <= z_max
fig.add_trace(go.Scatter(
    x=iot_obj.z[mask], y=iot_obj.f[mask], mode='lines',
    name='Fitted f(z)', line=dict(color='red', width=2)
))

fig.update_layout(
    template=template, xaxis_title='Income', yaxis_title='Density f(z)',
    xaxis=dict(range=[0, z_max])
)
fig.write_image(os.path.join(path, "income_dist_histogram.png"))

In [ ]:
# Plot distribution of log income with histogram and estimated density
data_pos = data[data[income_measure] > 0]
log_income = np.log(data_pos[income_measure])

hist_log, bin_edges_log = np.histogram(
    log_income, bins=20, weights=data_pos['s006'], density=True
)
bin_centers_log = (bin_edges_log[:-1] + bin_edges_log[1:]) / 2

# Transform fitted density: f_{log z}(y) = f(z) * z  (change of variables)
iot_obj = iot_all.iot[0]
mask_pos = iot_obj.z > 0
z_pos = iot_obj.z[mask_pos]
f_log = iot_obj.f[mask_pos] * z_pos

fig_log = go.Figure()
fig_log.add_trace(go.Bar(
    x=bin_centers_log, y=hist_log, name='Data', opacity=0.6,
    width=bin_edges_log[1] - bin_edges_log[0]
))
fig_log.add_trace(go.Scatter(
    x=np.log(z_pos), y=f_log, mode='lines',
    name='Fitted f(log z)', line=dict(color='red', width=2)
))
fig_log.update_layout(
    template=template, xaxis_title='Log Income', yaxis_title='Density',
)
fig_log.write_image(os.path.join(path, "income_dist_histogram_log.png"), scale=4)

In [ ]:
# Plots of theta(z) for each year/candidate
theta_plot = iot_all.plot(var="theta_z")
theta_plot.update_layout(
    template=template,
)
theta_plot.write_image(
            os.path.join(path, "theta.png"),
            scale=4
        )

In [ ]:
# Plot marginal tax rates for each year/candidate
mtr_plot = iot_all.plot(var="mtr")
mtr_plot.update_layout(
    template=template,
    xaxis_title="Income",
    yaxis_title="MTR",
)
mtr_plot.update_traces(
    line=dict(dash="dot", color="blue"),
    selector=dict(name="Obama 2015")
)
mtr_plot.update_traces(
    line=dict(dash="dot", color="red"),
    selector=dict(name="Romney 2012")
)
mtr_plot.update_traces(
    line=dict(dash="dash", color="blue"),
    selector=dict(name="Clinton 2016")
)
mtr_plot.update_traces(
    line=dict(dash="dash", color="red"),
    selector=dict(name="Trump 2016")
)
mtr_plot.update_traces(
    line=dict(dash="dashdot", color="blue"),
    selector=dict(name="Biden 2020")
)
mtr_plot.update_traces(
    line=dict(dash="dashdot", color="red"),
    selector=dict(name="Trump 2020")
)
mtr_plot.update_traces(
    line=dict(dash="solid", color="blue"),
    selector=dict(name="Harris 2024")
)
mtr_plot.update_traces(
    line=dict(dash="solid", color="red"),
    selector=dict(name="Trump 2024")
)
# mtr_plot.update_xaxes(range=[0, 850000])

mtr_plot.write_image(
            os.path.join(path, "mtr_all.png"),
            scale=4
        )

In [ ]:
# plots of g(z) for each year/candidate
gz_plot = iot_all.plot(var="g_z")
gz_plot.update_layout(
    template=template,
)
gz_plot.update_traces(
    line=dict(dash="dot", color="blue"),
    selector=dict(name="Obama 2015")
)
gz_plot.update_traces(
    line=dict(dash="dot", color="red"),
    selector=dict(name="Romney 2012")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="blue"),
    selector=dict(name="Clinton 2016")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="red"),
    selector=dict(name="Trump 2016")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="blue"),
    selector=dict(name="Biden 2020")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="red"),
    selector=dict(name="Trump 2020")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="blue"),
    selector=dict(name="Harris 2024")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="red"),
    selector=dict(name="Trump 2024")
)
# gz_plot.update_xaxes(range=[0, 850000])

gz_plot.write_image(
            os.path.join(path, "gz_all.png"),
            scale=4
        )

In [ ]:
# plots of g(z) for each year/candidate, numerical approach
gz_plot = iot_all.plot(var="g_z_numerical")
gz_plot.update_layout(
    template=template,
)
gz_plot.update_traces(
    line=dict(dash="dot", color="blue"),
    selector=dict(name="Obama 2015")
)
gz_plot.update_traces(
    line=dict(dash="dot", color="red"),
    selector=dict(name="Romney 2012")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="blue"),
    selector=dict(name="Clinton 2016")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="red"),
    selector=dict(name="Trump 2016")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="blue"),
    selector=dict(name="Biden 2020")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="red"),
    selector=dict(name="Trump 2020")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="blue"),
    selector=dict(name="Harris 2024")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="red"),
    selector=dict(name="Trump 2024")
)
# gz_plot.update_xaxes(range=[0, 850000])

gz_plot.write_image(
            os.path.join(path, "gz_numerical_all.png"),
            scale=4
        )

# Make the same plot but stop at 200_000
gz_plot.update_xaxes(range=[0, 200_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_all_200k.png"),
            scale=4
        )
gz_plot.update_xaxes(range=[0, 100_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_all_100k.png"),
            scale=4
        )

In [ ]:
# plots of g(z) for each year/candidate, numerical approach,
# HIGHlighting the Democrats
gz_plot = iot_all.plot(var="g_z_numerical")
gz_plot.update_layout(
    template=template,
)
gz_plot.update_traces(
    line=dict(dash="dot", color="blue"),
    selector=dict(name="Obama 2015")
)
gz_plot.update_traces(
    line=dict(dash="dot", color="gray"),
    selector=dict(name="Romney 2012")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="blue"),
    selector=dict(name="Clinton 2016")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="gray"),
    selector=dict(name="Trump 2016")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="blue"),
    selector=dict(name="Biden 2020")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="gray"),
    selector=dict(name="Trump 2020")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="blue"),
    selector=dict(name="Harris 2024")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="gray"),
    selector=dict(name="Trump 2024")
)
# gz_plot.update_xaxes(range=[0, 850000])

gz_plot.write_image(
            os.path.join(path, "gz_numerical_democrats.png"),
            scale=4
        )
gz_plot.update_xaxes(range=[0, 200_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_democrats_200k.png"),
            scale=4
        )
gz_plot.update_xaxes(range=[0, 100_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_democrats_100k.png"),
            scale=4
        )

In [ ]:
# plots of g(z) for each year/candidate, numerical approach
# HIGHlighting the Republicans
gz_plot = iot_all.plot(var="g_z_numerical")
gz_plot.update_layout(
    template=template,
)
gz_plot.update_traces(
    line=dict(dash="dot", color="gray"),
    selector=dict(name="Obama 2015")
)
gz_plot.update_traces(
    line=dict(dash="dot", color="red"),
    selector=dict(name="Romney 2012")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="gray"),
    selector=dict(name="Clinton 2016")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="red"),
    selector=dict(name="Trump 2016")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="gray"),
    selector=dict(name="Biden 2020")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="red"),
    selector=dict(name="Trump 2020")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="gray"),
    selector=dict(name="Harris 2024")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="red"),
    selector=dict(name="Trump 2024")
)
# gz_plot.update_xaxes(range=[0, 850000])

gz_plot.write_image(
            os.path.join(path, "gz_numerical_republicans.png"),
            scale=4
        )
gz_plot.update_xaxes(range=[0, 200_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_republicans_200k.png"),
            scale=4
        )
gz_plot.update_xaxes(range=[0, 100_000])
gz_plot.write_image(
            os.path.join(path, "gz_numerical_republicans_100k.png"),
            scale=4
        )

In [ ]:
# Show how MTRs vs tax base elasticity affecting g(z) for 2 candidates (separate plots,
# which will be put in a 2 panel figure)
fig = iot_all.JJZFig4(policy='Biden 2020', upper_bound=1_000_000)
fig.update_layout(
    template=template,
)
# fig.update_xaxes(range=[0, 850000])

fig.write_image(
            os.path.join(path, "composition_Biden2020_gz_1m.png"),
            scale=4
        )

In [ ]:
# Loop over values of epsilon and plot Biden under these alternative values"
eps_values = [0.2, 0.3, 0.4, 0.5, 0.6]
biden_eps_dict = {}
biden_eps_dict_numerical = {}
label_list = []
for i, v in enumerate(eps_values):
    label = r"$\varepsilon$ = " + str(v)
    iot_b = iot_user.iot_comparison(
        policies=[candidate_dict["Biden 2020"]["policy_path"]],
        baseline_policies=[candidate_dict["Biden 2020"]["baseline_path"]],
        labels=[label],
        years=[candidate_dict["Biden 2020"]["start_year"]],
        eti=v,
        data="CPS",
        income_measure=income_measure,
        dist_type="Pln",  # keep consistent with baseline results (iot_all)
    )
    label_list.append(label)
    biden_eps_dict[v] = iot_b.iot[0].df().g_z
    biden_eps_dict_numerical[v] = iot_b.iot[0].df().g_z_numerical

In [ ]:
# plot each g_z
label_dict = {}
redVSblue = n_colors('rgb(0, 0, 255)', 'rgb(255, 0, 0)', len(label_list), colortype = 'rgb')
for i, v in enumerate(label_list):
    label_dict["wide_variable_" + str(i)] = str(eps_values[i])#v
fig = px.line(
    x=iot_b.iot[0].df().z,
    y=[
        biden_eps_dict[0.2],
        biden_eps_dict[0.3],
        biden_eps_dict[0.4],
        biden_eps_dict[0.5],
        biden_eps_dict[0.6]
        ],
    color_discrete_sequence=redVSblue,
    labels=label_dict)
fig.for_each_trace(lambda t: t.update(name = label_dict[t.name], legendgroup = label_dict[t.name],
                                      hovertemplate = t.hovertemplate.replace(t.name, label_dict[t.name])))
fig.update_layout(
    template=template,
    xaxis_title="Wages and Salaries",
    yaxis_title=r"$g_z$",
    legend=dict(
        title="ETI value:",
        ),
)

# fig.update_xaxes(range=[0, 850000])
fig.write_image(
            os.path.join(path, "vary_ETI_Biden2020_gz.png"),
            scale=4
        )

# plot each g_z_numerical
label_dict = {}
redVSblue = n_colors('rgb(0, 0, 255)', 'rgb(255, 0, 0)', len(label_list), colortype = 'rgb')
for i, v in enumerate(label_list):
    label_dict["wide_variable_" + str(i)] = str(eps_values[i])#v
fig = px.line(
    x=iot_b.iot[0].df().z[50:],
    y=[
        biden_eps_dict_numerical[0.2][50:],
        biden_eps_dict_numerical[0.3][50:],
        biden_eps_dict_numerical[0.4][50:],
        biden_eps_dict_numerical[0.5][50:],
        biden_eps_dict_numerical[0.6][50:]
        ],
    color_discrete_sequence=redVSblue,
    labels=label_dict)
fig.for_each_trace(lambda t: t.update(name = label_dict[t.name], legendgroup = label_dict[t.name],
                                      hovertemplate = t.hovertemplate.replace(t.name, label_dict[t.name])))
fig.update_layout(
    template=template,
    xaxis_title="Wages and Salaries",
    yaxis_title=r"$g_z$",
    legend=dict(
        title="ETI value:",
        ),
)

# fig.update_xaxes(range=[0, 850000])
fig.write_image(
            os.path.join(path, "vary_ETI_Biden2020_gz_numerical.png"),
            scale=4
        )


In [ ]:
# Redo above with varying epsilon(z) according to some empirical studies
# Required modification of model
eti_dict = {
    "eti_values": [0.18, 0.106, 0.567, 1.83, 1.9],
    "knot_points": [30000, 75000, 250000, 2000000, 10000000]
}
iot_all_vary = iot_user.iot_comparison(
    policies=policies,
    baseline_policies=baseline_policies,
    labels=labels,
    years=years,
    eti=eti_dict,
    data="CPS",
    income_measure=income_measure,
    dist_type="Pln",  # keep consistent with baseline results (iot_all)
)


In [ ]:
# Plot how ETI varies with income
z_line = np.linspace(1, 1000000, 100000)
eti_dict = {
    "eti_values": [0.18, 0.106, 0.567, 1.83, 1.9],
    "knot_points": [30000, 75000, 250000, 2000000, 10000000]
}
eti_spl = UnivariateSpline(
    eti_dict["knot_points"], eti_dict["eti_values"], k=3, s=0
)
eti = eti_spl(z_line)
fig = px.line(x=z_line, y=eti, labels={"x": "Wages and Salaries", "y": r"$\varepsilon$"})
# add special markers without hoverinfo
fig.add_traces(
    go.Scatter(
        x=eti_dict["knot_points"][:-2], y=eti_dict["eti_values"][:-2], mode="markers", name="Gruber and Saez (2022)", hoverinfo="skip"
    )
)
# put legend at bottom
fig.update_layout(legend=dict(yanchor="bottom", y=0.7, xanchor="left", x=0.1))
fig.update_layout(
    template=template,
)
fig.write_image(
            os.path.join(path, "ETI_spline.png"),
            scale=4,
        )

In [ ]:
# plots of g(z) for each year/candidate, numerical approach
gz_plot = iot_all_vary.plot(var="g_z_numerical")
gz_plot.update_layout(
    template=template,
)
gz_plot.update_traces(
    line=dict(dash="dot", color="blue"),
    selector=dict(name="Obama 2015")
)
gz_plot.update_traces(
    line=dict(dash="dot", color="red"),
    selector=dict(name="Romney 2012")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="blue"),
    selector=dict(name="Clinton 2016")
)
gz_plot.update_traces(
    line=dict(dash="dash", color="red"),
    selector=dict(name="Trump 2016")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="blue"),
    selector=dict(name="Biden 2020")
)
gz_plot.update_traces(
    line=dict(dash="dashdot", color="red"),
    selector=dict(name="Trump 2020")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="blue"),
    selector=dict(name="Harris 2024")
)
gz_plot.update_traces(
    line=dict(dash="solid", color="red"),
    selector=dict(name="Trump 2024")
)
# gz_plot.update_xaxes(range=[0, 850000])

gz_plot.write_image(
            os.path.join(path, "gz_numerical_all_vary_eti.png"),
            scale=4
        )

In [ ]:
eti_dict = {
    "eti_values": [0.18, 0.106, 0.567, 1.83, 1.9],
    "knot_points": [30000, 75000, 250000, 2000000, 10000000]
}
iot_2023 = iot_user.iot_comparison(
    policies=[{}],
    baseline_policies=[None],
    labels=["2023 Law"],
    years=[2023],
    data="CPS",
    eti=eti_dict,
    income_measure=income_measure,
    dist_type="Pln",  # keep consistent with baseline results (iot_all)
)
fig = px.line(
    x=iot_2023.iot[0].df().z,
    y=iot_2023.iot[0].df().mtr,
)
fig.update_layout(
    template=template,
    xaxis_title="Wages and Salaries",
    yaxis_title=r"$T'(z)$",
)
fig.write_image(os.path.join(path, "MTR_2023.png"), scale=4)

In [ ]:
# JJZ Fig 4 decomposition for Obama 2015 (single-policy object)
# Use a separate variable so we don't clobber the full iot_all
iot_obama = iot_user.iot_comparison(
    policies=[policies[0]],
    baseline_policies=[baseline_policies[0]],
    mtr_smoother="kreg",
    labels=[labels[0]],
    years=[years[0]],
    data="CPS",
    income_measure=income_measure,
    dist_type="Pln",  # keep consistent with baseline results (iot_all)
)
fig = iot_obama.JJZFig4(policy="Obama 2015")
fig.update_layout(template=template)
fig.show()

In [ ]:
# Do experiment where hold constant g(z) - pick a candidate as baseline - then plot epsilon(z)
# that would recover those g(z) given the tax rates of each candidate
# one plot with epsilon(z) for each candidate
# Will pick Trump and Clinton (2016) for example

from iot.inverse_optimal_tax import find_eti

candidate_pairs = [
    (0, 1, ["Obama 2015", "Romney 2012"], "romney_obama_g_z_numerical.png"),
    (2, 3, ["Clinton 2016", "Trump 2016"], "trump_clinton_g_z_numerical.png"),
    (4, 5, ["Biden 2020", "Trump 2020"],   "trump_biden_g_z_numerical.png"),
    (6, 7, ["Harris 2024", "Trump 2024"],  "trump_harris_g_z_numerical.png"),
]

for idx1, idx2, candidate_name, filename in candidate_pairs:
    fig = px.line(
        x=iot_all.iot[idx1].df().z[10:],
        y=[iot_all.iot[idx1].df().g_z_numerical[10:], iot_all.iot[idx2].df().g_z_numerical[10:]],
        labels={"x": "Wages and Salaries", "y": r"$g_z$"},
    )
    fig.update_layout(template=template, legend=dict(title="Candidate:"))
    label_dict = {"wide_variable_0": candidate_name[0], "wide_variable_1": candidate_name[1]}
    fig.for_each_trace(lambda t: t.update(name=label_dict[t.name], legendgroup=label_dict[t.name],
                                          hovertemplate=t.hovertemplate.replace(t.name, label_dict[t.name])))
    fig.write_image(os.path.join(path, filename), scale=4)

In [ ]:
# fig:trump_eti — Trump 2016 with Clinton's g_z, back out ETI
clinton_gz = iot_all.iot[2].g_z

# (a) boundary at z0 with eps_0 = 0.25
eti_trump_z0 = find_eti(iot_all.iot[3], g_z=clinton_gz, eti_0=0.25, boundary="z0")
# (b) transversality condition
eti_trump_inf = find_eti(iot_all.iot[3], g_z=clinton_gz, boundary="inf")

z = iot_all.iot[3].z

# --- Main-text figure (fig:trump_eti, images/eti_trump.png) ---
# Show ONLY the initial-condition (eti_0) curve. Holding Clinton's g(z)
# fixed, Trump's uniformly lower MTRs should imply a *higher* elasticity;
# the z0 solution satisfies this benchmark (rises above 0.25) while the
# transversality solution fails it (drifts below 0.25). See the appendix
# discussion at fig:eti_trump_compare. We therefore adopt the z0 curve
# for the main text and plot a single curve.
#
# Two versions are written: the paper's figure (fig:trump_eti,
# images/eti_trump.png) caps the x-axis at 500k, and eti_trump_1m.png
# extends it to 1M.
x_lo = 1_000
for x_hi, fname in [(500_000, "eti_trump.png"), (1_000_000, "eti_trump_1m.png")]:
    fig_trump = go.Figure()
    fig_trump.add_trace(go.Scatter(
        x=z, y=eti_trump_z0, mode="lines",
        name=r"Implied ETI",
        line=dict(color="blue", width=2.5),
    ))
    fig_trump.add_hline(
        y=0.25, line_dash="dot", line_color="gray",
        annotation_text="ε = 0.25", annotation_position="bottom right",
    )

    # Size the y-axis to the data actually shown (plus the reference line)
    # instead of a fixed +/-0.5 window, which left most of the figure blank
    # since the curve only ever ranges from about 0.25 to 0.37.
    x_mask = (z >= x_lo) & (z <= x_hi)
    y_vals = np.concatenate([eti_trump_z0[x_mask], [0.25]])
    y_pad = 0.3 * (y_vals.max() - y_vals.min())
    y_range = [y_vals.min() - y_pad, y_vals.max() + y_pad]

    fig_trump.update_layout(
        template=template,
        xaxis_title="Income",
        yaxis_title=r"Implied ETI",
        yaxis=dict(range=y_range),
        xaxis=dict(range=[x_lo, x_hi]),
    )
    fig_trump.write_image(os.path.join(path, fname), scale=4)
fig_trump.show()

# --- Appendix comparison figure (fig:eti_trump_compare,
# images/eti_trump_compare_bc.png) — keeps BOTH boundary conditions,
# since that figure's whole purpose is to compare the two solutions. ---
fig_trump_compare = go.Figure()
fig_trump_compare.add_trace(go.Scatter(
    x=z, y=eti_trump_z0, mode="lines",
    name=r"ETI_0 = 0.25",
    line=dict(color="blue", width=2),
))
fig_trump_compare.add_trace(go.Scatter(
    x=z, y=eti_trump_inf, mode="lines",
    name="Transversality",
    line=dict(color="red", width=2, dash="dash"),
))
fig_trump_compare.add_hline(
    y=0.25, line_dash="dot", line_color="gray",
    annotation_text="ε = 0.25", annotation_position="top left",
)
fig_trump_compare.update_layout(
    template=template,
    xaxis_title="Income",
    yaxis_title=r"Implied ETI",
    legend=dict(title="Boundary condition"),
    yaxis=dict(range=[-0.5, 0.5]),
    xaxis=dict(range=[1_000, 500_000]),
)
fig_trump_compare.write_image(
    os.path.join(path, "eti_trump_compare_bc.png"), scale=4
)
fig_trump_compare.show()

In [ ]:
# fig:eti_utilitarian — All candidates, g_z = 1, back out ETI
utilitarian_gz = np.ones_like(iot_all.iot[0].g_z)

style_map = {
    "Obama 2015":   dict(color="blue",  dash="dot"),
    "Romney 2012":  dict(color="red",   dash="dot"),
    "Clinton 2016": dict(color="blue",  dash="dash"),
    "Trump 2016":   dict(color="red",   dash="dash"),
    "Biden 2020":   dict(color="blue",  dash="dashdot"),
    "Trump 2020":   dict(color="red",   dash="dashdot"),
    "Harris 2024":  dict(color="blue",  dash="solid"),
    "Trump 2024":   dict(color="red",   dash="solid"),
}

# --- Single-curve-set utilitarian figure (fig:eti_utilitarian,
# images/eti_utilitarian.png) ---
# Use the SAME boundary condition adopted for the main Trump figure: the
# initial condition at eps_0 = 0.25 (boundary="z0"). The transversality
# solution fails the "lower MTRs => higher implied ETI" benchmark (see
# fig:eti_trump_compare), so we do not plot it here. One boundary
# condition only, all candidates.
fig_util = go.Figure()
x_lo, x_hi = 1_000, 500_000
eti_util_all = []
for i, label in enumerate(labels):
    eti_util = find_eti(iot_all.iot[i], g_z=utilitarian_gz, eti_0=0.25, boundary="z0")
    eti_util_all.append(eti_util)
    sty = style_map.get(label, dict(color="black", dash="solid"))
    fig_util.add_trace(go.Scatter(
        x=iot_all.iot[i].z, y=eti_util, mode="lines",
        name=label,
        line=dict(color=sty["color"], dash=sty["dash"], width=2),
    ))

# The y-axis is sized to the data (plus a small pad) instead of a fixed
# +/-0.5 window. The x-axis uses the same linear scale and range as the
# other ETI figures (e.g. fig:eti_trump_compare) for consistency.
z_ref = iot_all.iot[0].z
x_mask = (z_ref >= x_lo) & (z_ref <= x_hi)
y_vals = np.concatenate([y[x_mask] for y in eti_util_all])
y_pad = 0.1 * (y_vals.max() - y_vals.min())
y_range = [y_vals.min() - y_pad, y_vals.max() + y_pad]

fig_util.update_layout(
    template=template,
    xaxis_title="Income",
    yaxis_title=r"$\text{Implied } \varepsilon(z)$",
    legend=dict(title="Candidate"),
    # Force plain/scientific tick labels rather than plotly's default
    # SI-prefix notation (e.g. "400µ"), which reads poorly in a paper
    # figure when the visible range happens to be small.
    yaxis=dict(range=y_range, exponentformat="power"),
    xaxis=dict(range=[x_lo, x_hi]),
)
fig_util.write_image(os.path.join(path, "eti_utilitarian.png"), scale=4)
fig_util.show()

In [ ]:
iot_all.pctile_table()

In [ ]:
iot_all.pctile_table(table_format="csv")